# 02. Build Brain Assets

TRIBE raw output을 세그먼트별 downsampled color array로 바꿔 프론트에서 바로 읽을 수 있는 JSON으로 저장한다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, numpy as np

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/tribe-v2-student-reaction')
RAW_DIR = ROOT / 'outputs' / 'raw'
ASSET_DIR = ROOT / 'outputs' / 'assets'
PREPARED_DIR = ROOT / 'outputs' / 'prepared'
ASSET_DIR.mkdir(parents=True, exist_ok=True)
PILOT_DATES = ['2026-02-02', '2026-02-09', '2026-02-24']

In [ ]:
def normalize_segment(pred):
    pred = pred.astype('float32')
    pred = pred - pred.min()
    denom = max(float(pred.max()), 1e-6)
    return (pred / denom).round(4).tolist()

def split_hemispheres(pred):
    midpoint = pred.shape[0] // 2
    if midpoint * 2 != pred.shape[0]:
        raise ValueError(f'Expected even vertex count, got {pred.shape[0]}')
    return pred[:midpoint], pred[midpoint:]

for date in PILOT_DATES:
    raw = np.load(RAW_DIR / f'{date}-tribe-raw.npz')['preds']
    prepared = json.loads((PREPARED_DIR / date / 'segments.json').read_text(encoding='utf-8'))
    payload = {
        'lecture_date': date,
        'mesh_version': 'pilot-v1',
        'vertex_count': int(raw.shape[1] // 2),
        'segments': [],
    }
    for idx, segment in enumerate(prepared):
        left_pred, right_pred = split_hemispheres(raw[idx])
        payload['segments'].append({
            'segment_id': segment['segment_id'],
            'start_time': segment['start_time'],
            'end_time': segment['end_time'],
            'hemispheres': {
                'left': normalize_segment(left_pred),
                'right': normalize_segment(right_pred),
            },
        })
    (ASSET_DIR / f'{date}-segment-colors.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print('saved', date)
